# SDE-Net direct multi-horizon post-hoc con label STGAN

Questo notebook **non riaddestra SDE-Net o STGAN**. Legge l'unico `predictions.csv` del modello diretto t+1,…,t+6, rimuove dall'analisi fisica i dropout solari regionali e la loro ora di recovery, ricalcola il top-1% STGAN sui soli punti validi e rigenera l'analisi normal/rare.

La decisione STGAN originale resta disponibile come `detector_is_anomaly_original`; `anomaly_group` usa invece la graduatoria pulita. I punti rimossi non diventano normali: sono esportati separatamente come anomalie di qualità. `event_group` non viene creato.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / 'physiq_pv').is_dir():
    for parent in Path.cwd().resolve().parents:
        if (parent / 'physiq_pv').is_dir():
            ROOT = parent
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from physiq_pv.experiments import sde_pipeline as pipe
from physiq_pv.reporting.pointwise_detector_posthoc import build_pointwise_detector_evaluation
print('repo root:', ROOT)

## 1. Percorsi e configurazione

Sul server si possono sovrascrivere i percorsi con `STGAN_SEED_DIR`, `SDE_MULTIHORIZON_PREDICTIONS`, `PVGIS_2019_FILE` e `STGAN_POSTHOC_ROOT`.

In [ ]:
FORECAST_HORIZONS = pipe.FORECAST_HORIZONS
STGAN_SEED = 20
STGAN_SEED_DIR = Path(os.environ.get(
    'STGAN_SEED_DIR', ROOT / 'outputs' / 'pvgis_stgan' / 'paper_reference' / f'seed_{STGAN_SEED}'
)).resolve()
STGAN_SCORES = STGAN_SEED_DIR / 'anomaly_scores.csv'
PVGIS_2019 = Path(os.environ.get(
    'PVGIS_2019_FILE', ROOT / 'data' / 'pvgis' / 'piedmont_pvgis_2019.nc'
)).resolve()
STGAN_PREPARED_MANIFEST = (
    ROOT / 'outputs' / 'pvgis_stgan' / 'prepared' / 'manifest.csv'
).resolve()
PVGIS_QUALITY_CANDIDATES = (PVGIS_2019, STGAN_PREPARED_MANIFEST)
PVGIS_QUALITY_SOURCE = next(
    (path for path in PVGIS_QUALITY_CANDIDATES if path.is_file()),
    PVGIS_2019,
)
CLEAN_TOP_K_PERCENT = 1.0

# È lo stesso run diretto del notebook SDE-Net principale. STGAN sostituisce
# le label MTGFlow soltanto nella valutazione post-hoc.
BASE_CONFIG = {**pipe.DEFAULT_CONFIG,
    'name': 'paper_faithful_gaussian_detector_mtgflow_ep60',
    'forecast_horizons': ','.join(map(str, FORECAST_HORIZONS)),
    'epochs': 60, 'batch_size': 16, 'lr': 1e-4, 'lr_g': 1e-2,
    'dropout': 0.0, 'train_normal_only': False, 'anomaly_source': 'detector',
    'ood_smoke_test': True, 'sde_sigma_warmup_epochs': 30,
    'irradiance_loss_weight': 0.1, 'detector_regional_quantile': 0.975,
}
SDE_PREDICTIONS = Path(os.environ.get(
    'SDE_MULTIHORIZON_PREDICTIONS',
    ROOT / pipe.make_out_dir(BASE_CONFIG) / 'predictions.csv',
)).resolve()
EVALUATION_DIR = Path(os.environ.get(
    'STGAN_POSTHOC_ROOT',
    ROOT / 'outputs' / f'sde_stgan_direct_multihorizon_seed{STGAN_SEED}_quality_filtered'
)).resolve()
MIN_MATCH_FRACTION = 0.90
RUN_RELABEL = not (EVALUATION_DIR / 'evaluation_source.json').is_file()
RUN_ANALYSIS = True
ALLOW_OVERWRITE = False

print('horizons:', FORECAST_HORIZONS)
print('STGAN scores:', STGAN_SCORES)
print('PVGIS quality source:', PVGIS_QUALITY_SOURCE)
print('SDE direct predictions:', SDE_PREDICTIONS)
print('evaluation output:', EVALUATION_DIR)

In [ ]:
missing = [path for path in (STGAN_SCORES, SDE_PREDICTIONS) if not path.is_file()]
if missing:
    raise FileNotFoundError('File mancanti:\n' + '\n'.join(map(str, missing)))
if not PVGIS_QUALITY_SOURCE.is_file():
    raise FileNotFoundError(
        'Nessuna sorgente PVGIS per il controllo qualità. Percorsi provati:\n'
        + '\n'.join(map(str, PVGIS_QUALITY_CANDIDATES))
    )
stgan_header = set(pd.read_csv(STGAN_SCORES, nrows=0).columns)
required_stgan = {'location', 'timestamp', 'anomaly_score', 'threshold', 'is_anomaly'}
if not required_stgan <= stgan_header:
    raise ValueError(f'Colonne STGAN mancanti: {sorted(required_stgan - stgan_header)}')
sde_header = set(pd.read_csv(SDE_PREDICTIONS, nrows=0).columns)
required_sde = {
    'location', 'timestamp', 'issue_timestamp', 'horizon_hours', 'y_true',
    'solar_irradiance_poa_target',
}
if not required_sde <= sde_header or not ({'y_pred', 'y_pred_mean'} & sde_header):
    raise ValueError(f'Schema SDE-Net diretto non compatibile: {sorted(sde_header)}')
saved_horizons = tuple(sorted(pd.read_csv(
    SDE_PREDICTIONS, usecols=['horizon_hours']
)['horizon_hours'].drop_duplicates().astype(int)))
if saved_horizons != FORECAST_HORIZONS:
    raise ValueError(f'Orizzonti salvati {saved_horizons}, attesi {FORECAST_HORIZONS}.')
print('OK: CSV diretto t+1,...,t+6, STGAN e PVGIS 2019 disponibili')

## 2. Join puntuale STGAN → canali diretti SDE-Net

Le label MTGFlow già presenti vengono rimosse. Il filtro identifica senza date hard-coded gli azzeramenti regionali isolati di direct, diffuse, sun height e produzione PV, esclude anche la recovery t+1 e ricostruisce l'esatto budget globale top-1% del paper sulle coordinate valide.

In [ ]:
if RUN_RELABEL:
    RELABEL_RESULT = build_pointwise_detector_evaluation(
        SDE_PREDICTIONS, STGAN_SCORES, EVALUATION_DIR,
        detector_name='stgan', min_match_fraction=MIN_MATCH_FRACTION,
        allow_overwrite=ALLOW_OVERWRITE,
        pvgis_quality_source=PVGIS_QUALITY_SOURCE,
        clean_top_k_percent=CLEAN_TOP_K_PERCENT,
    )
else:
    print("RUN_RELABEL=False: uso l'output evaluation-only esistente.")

metadata_path = EVALUATION_DIR / 'evaluation_source.json'
if not metadata_path.is_file():
    raise FileNotFoundError(metadata_path)
AUDIT = json.loads(metadata_path.read_text(encoding='utf-8'))
display(pd.DataFrame([AUDIT])[[
    'detector', 'forecast_mode', 'horizons_hours', 'source_prediction_rows',
    'detector_matched_rows_before_quality_filter', 'matched_rows',
    'excluded_unmatched_rows', 'excluded_data_quality_rows', 'match_fraction',
    'solar_dropout_timestamps', 'data_quality_timestamps', 'clean_top_k_percent',
    'eligible_detector_coordinates', 'data_quality_detector_coordinates',
    'original_detector_anomalies', 'clean_detector_anomalies',
    'normal_rows', 'rare_rows',
]])

## 3. Audit delle anomalie di qualità

Queste coordinate vengono conservate come anomalie del dato, ma non partecipano alla graduatoria top-1% fisica né alle metriche normal/rare.

In [ ]:
QUALITY_ISSUES = pd.read_csv(
    EVALUATION_DIR / 'pvgis_data_quality_issues.csv', parse_dates=['timestamp', 'source_dropout_timestamp']
)
QUALITY_PREDICTIONS = pd.read_csv(
    EVALUATION_DIR / 'data_quality_predictions.csv',
    usecols=lambda c: c in {
        'location', 'timestamp', 'horizon_hours', 'detector_is_anomaly_original'
    },
)
display(QUALITY_ISSUES)
display(QUALITY_PREDICTIONS.groupby('horizon_hours').agg(
    excluded_rows=('location', 'size'),
    originally_flagged=('detector_is_anomaly_original', 'sum'),
))

In [ ]:
joined_path = EVALUATION_DIR / 'predictions.csv'
joined = pd.read_csv(joined_path, usecols=lambda c: c in {
    'location', 'timestamp', 'horizon_hours', 'anomaly_group', 'event_group',
    'detector_is_anomaly', 'detector_is_anomaly_original', 'detector_anomaly_score',
    'solar_irradiance_poa_target'
})
assert 'anomaly_group' in joined
assert 'event_group' not in joined
assert tuple(sorted(joined['horizon_hours'].unique())) == FORECAST_HORIZONS
assert not joined.duplicated(['location', 'timestamp', 'horizon_hours']).any()
print('OK: t+1,...,t+6 usano il top-1% STGAN ricalcolato sui target validi')

In [ ]:
clean_labels = joined.loc[
    joined['horizon_hours'].eq(1) & joined['solar_irradiance_poa_target'].gt(10.0)
].copy()
clean_labels['timestamp'] = pd.to_datetime(clean_labels['timestamp'])
CLEAN_TIMESTAMP_RANKING = clean_labels.groupby('timestamp', as_index=False).agg(
    n_scored=('location', 'size'),
    n_anomalies=('detector_is_anomaly', 'sum'),
    anomaly_score_mean=('detector_anomaly_score', 'mean'),
    anomaly_score_max=('detector_anomaly_score', 'max'),
)
CLEAN_TIMESTAMP_RANKING['anomaly_share'] = (
    CLEAN_TIMESTAMP_RANKING['n_anomalies'] / CLEAN_TIMESTAMP_RANKING['n_scored']
)
CLEAN_TIMESTAMP_RANKING = CLEAN_TIMESTAMP_RANKING.sort_values(
    ['n_anomalies', 'anomaly_score_max'], ascending=False
)
CLEAN_DAILY_RANKING = (
    CLEAN_TIMESTAMP_RANKING.assign(day=lambda frame: frame['timestamp'].dt.floor('D'))
    .groupby('day', as_index=False)
    .agg(n_anomalies=('n_anomalies', 'sum'), peak_share=('anomaly_share', 'max'),
         peak_score=('anomaly_score_max', 'max'))
    .sort_values(['n_anomalies', 'peak_score'], ascending=False)
)
CLEAN_TIMESTAMP_RANKING.to_csv(EVALUATION_DIR / 'stgan_clean_daytime_timestamp_ranking.csv', index=False)
CLEAN_DAILY_RANKING.to_csv(EVALUATION_DIR / 'stgan_clean_daytime_daily_ranking.csv', index=False)
display(Markdown('### Nuovi timestamp STGAN diurni più anomali'))
display(CLEAN_TIMESTAMP_RANKING.head(20))
display(Markdown('### Nuovi giorni STGAN diurni più anomali'))
display(CLEAN_DAILY_RANKING.head(20))

## 4. Analisi post-hoc e suite per bin separate per t+1 e t+6

In [ ]:
DETAILED_HORIZONS = (1, 6)
REQUIRED_FIGURE_PREFIXES = ('mae_', 'rmse_', 'nmpil_', 'picp_', 'clc_')
FULL_POSTHOC_FIGURES = {}
for horizon_hours in DETAILED_HORIZONS:
    detail_out = EVALUATION_DIR / 'posthoc_by_horizon' / f't_plus_{horizon_hours}'
    analysis_command = pipe.build_analysis_command(
        str(detail_out), BASE_CONFIG, predictions=str(joined_path),
        horizon_hours=horizon_hours,
    )
    if RUN_ANALYSIS:
        subprocess.run(analysis_command, check=True, cwd=ROOT)
    elif not (detail_out / 'daytime_bin_anomaly_metrics.csv').is_file():
        raise FileNotFoundError(
            f'RUN_ANALYSIS=False ma manca il report per t+{horizon_hours}: {detail_out}'
        )
    figures = pipe.build_posthoc_figures(
        str(detail_out), horizon_hours=horizon_hours
    )
    missing = [
        prefix for prefix in REQUIRED_FIGURE_PREFIXES
        if not any(name.startswith(prefix) for name in figures)
    ]
    if missing:
        raise RuntimeError(
            f'Suite post-hoc STGAN incompleta per t+{horizon_hours}: {missing}'
        )
    FULL_POSTHOC_FIGURES[horizon_hours] = figures
    print(
        f't+{horizon_hours}: {len(figures)} figure STGAN per bin in {detail_out}'
    )

In [ ]:
POSTHOC_PATHS = pipe.build_direct_multihorizon_posthoc(EVALUATION_DIR)
DIRECT_METRICS = pd.read_csv(POSTHOC_PATHS['metrics'])
display(DIRECT_METRICS)
print({name: str(path) for name, path in POSTHOC_PATHS.items()})

display(Markdown('## Confronto multi-orizzonte STGAN'))
for key in ('error_figure', 'boxplot_figure', 'histogram_figure', 'prediction_figure'):
    display(Image(filename=str(POSTHOC_PATHS[key])))
for horizon_hours, figures in FULL_POSTHOC_FIGURES.items():
    display(Markdown(f'## Suite STGAN per bin — t+{horizon_hours}'))
    for name, path in sorted(figures.items()):
        display(Markdown(f'**{name}**'))
        display(Image(filename=str(path)))

## Interpretazione

Le curve confrontano MAE e RMSE sui punti validi che il **top-1% STGAN pulito** classifica normali o anomali per la stessa località e lo stesso timestamp target. Le decisioni originali e i dropout restano negli output di audit. Le suite dettagliate per bin sono calcolate separatamente per t+1 e t+6 e includono MAE, RMSE, NMPIL, PICP e CLC. Questa è un'analisi post-hoc esplorativa: non modifica il training e non interpreta più i dropout come eventi meteorologici.